# LSTM Image Captioning

Training dan eksperimen model LSTM untuk image captioning.

---

## 1. Setup & Imports

In [1]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Resolve project root: works when Jupyter is started from project root OR from src/lstm/
ROOT_DIR = Path('.').resolve()
if not (ROOT_DIR / 'src').exists():
    ROOT_DIR = (Path('..') / '..').resolve()
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Project root : {ROOT_DIR}')
print(f'src on path  : {SRC_DIR}')

import tensorflow as tf
import keras
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
try:
    from tensorflow.keras.preprocessing.text import tokenizer_from_json
except ImportError:
    # Fallback for versions where tokenizer_from_json is not available
    from tensorflow.keras.preprocessing.text import Tokenizer
    def tokenizer_from_json(json_string):
        """Load tokenizer from JSON string (fallback implementation)"""
        import json as json_module
        config = json_module.loads(json_string)
        tokenizer = Tokenizer(**config.get('config', {}))
        tokenizer.word_index = config.get('word_index', {})
        tokenizer.word_counts = config.get('word_counts', {})
        tokenizer.word_docs = config.get('word_docs', {})
        tokenizer.document_count = config.get('document_count', 0)
        return tokenizer

print(f'Keras version: {keras.__version__}')


Project root : C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption
src on path  : C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\src
Keras version: 3.14.0


In [2]:
# Config
EMBED_DIM  = 256
HIDDEN_DIM = 512
EPOCHS     = 20
BATCH_SIZE = 64

# Paths
DATA_DIR       = ROOT_DIR / 'data'
CAPTIONS_PATH  = DATA_DIR / 'captions.txt'
FEATURES_PATH  = DATA_DIR / 'flickr8k_features.npy'
TOKENIZER_PATH = DATA_DIR / 'tokenizer.json'
IMAGES_DIR     = DATA_DIR / 'Images'
WEIGHTS_DIR    = ROOT_DIR / 'weights'
WEIGHTS_DIR.mkdir(exist_ok=True)

print('Paths:')
for label, p in [
    ('captions',  CAPTIONS_PATH),
    ('features',  FEATURES_PATH),
    ('tokenizer', TOKENIZER_PATH),
    ('images',    IMAGES_DIR),
    ('weights',   WEIGHTS_DIR),
]:
    status = 'OK' if p.exists() else 'NOT FOUND'
    print(f'  {label:<12}: {p}  [{status}]')


Paths:
  captions    : C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\data\captions.txt  [OK]
  features    : C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\data\flickr8k_features.npy  [OK]
  tokenizer   : C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\data\tokenizer.json  [OK]
  images      : C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\data\Images  [OK]
  weights     : C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\weights  [OK]


---
## 2. Feature Extraction

Ekstrak fitur CNN dari seluruh gambar Flickr8k menggunakan InceptionV3 dan simpan ke data/flickr8k_features.npy.
Jika file sudah ada, langkah ini akan dilewati.

In [3]:
from shared.image_utils import extract_features

if FEATURES_PATH.exists():
    print(f'Features already exist at {FEATURES_PATH}')
    features = np.load(str(FEATURES_PATH), allow_pickle=True).item()
else:
    assert IMAGES_DIR.exists(), f'Images directory not found: {IMAGES_DIR}'
    image_paths = sorted(IMAGES_DIR.glob('*.jpg'))
    assert len(image_paths) > 0, f'No .jpg images found in {IMAGES_DIR}'
    print(f'Found {len(image_paths)} images — starting extraction...')

    encoder = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
    features = extract_features(
        paths=[str(p) for p in image_paths],
        output_path=str(FEATURES_PATH),
        batch_size=64,
        encoder=encoder,
    )

sample_key = next(iter(features))
print(f'Loaded {len(features)} features. Sample shape: {features[sample_key].shape}')


Features already exist at C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\data\flickr8k_features.npy
Loaded 4112 features. Sample shape: (2048,)


---
## 3. Caption Preprocessing

Load captions.txt, bersihkan, bangun tokenizer, dan hitung max_length.
Tokenizer disimpan ke data/tokenizer.json.

In [4]:
import csv as _csv

from shared.preprocessing import (
    load_captions,
    clean_and_wrap_captions,
    build_tokenizer,
    calculate_max_length,
)

assert CAPTIONS_PATH.exists(), f'captions.txt not found: {CAPTIONS_PATH}'

# ── Detect file format from first non-header line ─────────────────────────
with open(str(CAPTIONS_PATH), 'r', encoding='utf-8') as _f:
    _header = _f.readline()
    _sample = _f.readline()

_delim = '\t' if '\t' in _sample else ','
print(f'Detected delimiter: {"TAB" if _delim == chr(9) else "COMMA"}')

caption_mapping = load_captions(str(CAPTIONS_PATH))   # works when tab-delimited

# ── Fallback: re-parse with comma delimiter if tab parsing returned nothing ─
if len(caption_mapping) == 0:
    print('Tab-delimited parsing returned 0 captions — retrying with comma delimiter...')
    _mapping = {}
    with open(str(CAPTIONS_PATH), 'r', encoding='utf-8', newline='') as _f:
        _rows = _csv.reader(_f, delimiter=',')
        next(_rows, None)                              # skip header
        for _row in _rows:
            if len(_row) < 2:
                continue
            _img_id  = _row[0].strip()
            _caption = _row[1].strip()
            if _img_id and _caption:
                _mapping.setdefault(_img_id, []).append(_caption)
    caption_mapping = _mapping
    print(f'Comma-delimited fallback loaded {len(caption_mapping)} images.')

assert len(caption_mapping) > 0, (
    'caption_mapping is still empty after both tab and comma parsing.\n'
    f'Check that {CAPTIONS_PATH} has format: image_id<delim>caption'
)

print(f'Loaded captions for {len(caption_mapping)} images.')

cleaned_mapping = clean_and_wrap_captions(caption_mapping)
tokenizer, all_captions = build_tokenizer(cleaned_mapping, str(TOKENIZER_PATH))
max_length = calculate_max_length(all_captions)
vocab_size = len(tokenizer.word_index) + 1

print(f'vocab_size    : {vocab_size}')
print(f'max_length    : {max_length}')
print(f'total captions: {len(all_captions)}')


Detected delimiter: COMMA
Tab-delimited parsing returned 0 captions — retrying with comma delimiter...
Comma-delimited fallback loaded 8091 images.
Loaded captions for 8091 images.
Ukuran Vocabulary: 8832
Tokenizer berhasil disimpan di C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\data\tokenizer.json
Maximum Caption Length: 38
vocab_size    : 8832
max_length    : 38
total captions: 40455


---
## 4. Prepare Training Data

Buat pasangan input-output (cnn_feature, caption_prefix) menggunakan teacher forcing.

In [5]:
from lstm.train import prepare_sequences

print('Preparing training sequences...')
X, y = prepare_sequences(features, tokenizer, cleaned_mapping, max_length, EMBED_DIM)

cnn_features  = np.array([x[0] for x in X])   # (N, 2048)
cap_sequences = np.array([x[1] for x in X])   # (N, max_length)

print(f'Training samples    : {len(y)}')
print(f'cnn_features.shape  : {cnn_features.shape}')
print(f'cap_sequences.shape : {cap_sequences.shape}')


Preparing training sequences...
Training samples    : 243261
cnn_features.shape  : (243261, 2048)
cap_sequences.shape : (243261, 38)


---
## 5. Training Experiments

6 variasi kombinasi jumlah layer LSTM dan hidden dimension:

| Variasi | LSTM Layers | Hidden Dim |
|---------|-------------|------------|
| 1 | 1 | 128 |
| 2 | 2 | 128 |
| 3 | 3 | 128 |
| 4 | 1 | 512 |
| 5 | 2 | 512 |
| 6 | 3 | 512 |

In [6]:
def build_model_variant(
    vocab_size: int,
    embed_dim: int,
    max_length: int,
    hidden_dim: int,
    n_layers: int,
    cnn_feature_dim: int = 2048,
) -> Model:
    """Keras model dengan stacked LSTM layers yang dapat dikonfigurasi."""
    # CNN feature branch
    cnn_input = Input(shape=(cnn_feature_dim,), name='cnn_input')
    cnn_proj  = Dense(embed_dim, activation='relu', name='cnn_proj')(cnn_input)
    cnn_proj  = Reshape((1, embed_dim), name='cnn_reshape')(cnn_proj)

    # Caption sequence branch
    cap_input = Input(shape=(max_length,), name='cap_input')
    cap_embed = Embedding(vocab_size, embed_dim, mask_zero=True, name='embedding')(cap_input)

    # Pre-inject: prepend CNN feature vector ke awal sequence -> (max_length+1, embed_dim)
    merged = Concatenate(axis=1, name='pre_inject')([cnn_proj, cap_embed])

    # Stacked LSTM layers
    x = merged
    for i in range(n_layers):
        return_seq = (i < n_layers - 1)  # semua layer kecuali terakhir return sequences
        x = LSTM(hidden_dim, return_sequences=return_seq, name=f'lstm_{i + 1}')(x)

    outputs = Dense(vocab_size, activation='softmax', name='output')(x)

    model = Model(inputs=[cnn_input, cap_input], outputs=outputs)
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


In [ ]:
VARIATIONS = [
    (1, 128),
    (2, 128),
    (3, 128),
    (1, 512),
    (2, 512),
    (3, 512),
]

all_histories = {}

for n_layers, hidden_dim in VARIATIONS:
    variant_name = f'{n_layers}layers_{hidden_dim}hidden'
    weights_path = WEIGHTS_DIR / f'lstm_{variant_name}.weights.h5'

    print('\n' + '=' * 60)
    print(f'  Variation : {variant_name}')
    print(f'  Weights   : {weights_path}')
    print('=' * 60)

    model = build_model_variant(vocab_size, EMBED_DIM, max_length, hidden_dim, n_layers)
    model.summary()

    history = model.fit(
        [cnn_features, cap_sequences],
        y,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        verbose=1,
    )

    model.save_weights(str(weights_path))
    print(f'Saved -> {weights_path}')
    all_histories[variant_name] = history.history

    # Per-variation loss/accuracy plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Training History — {variant_name}', fontsize=13)

    axes[0].plot(history.history['loss'],     label='Train')
    axes[0].plot(history.history['val_loss'], label='Val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['accuracy'],     label='Train')
    axes[1].plot(history.history['val_accuracy'], label='Val')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(WEIGHTS_DIR / f'plot_{variant_name}.png'), dpi=100, bbox_inches='tight')
    plt.show()

print('\nAll variations trained.')



  Variation : 1layers_128hidden
  Weights   : C:\Users\Mahesa\OneDrive\ITB\Coding\College\Academic\IF\Smt-6\machine-learning\tugas\tugas-besar-2\PixelToCaption\weights\lstm_1layers_128hidden.weights.h5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ cnn_input           │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cap_input           │ (None, 38)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_proj (Dense)    │ (None, 256)       │    524,544 │ cnn_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 38, 256)   │  2,260,992 │ cap_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 38)        │          0 │ cap_input[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_reshape         │ (None, 1, 256)    │          0 │ cnn_proj[0][0]    │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, 38, 1)     │          0 │ not_equal[0][0]   │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zeros_like          │ (None, 38, 256)   │          0 │ embedding[0][0]   │
│ (ZerosLike)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ones_like           │ (None, 1, 256)    │          0 │ cnn_reshape[0][0] │
│ (OnesLike)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ logical_or          │ (None, 38, 256)   │          0 │ expand_dims[0][0… │
│ (LogicalOr)         │                   │            │ zeros_like[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 39, 256)   │          0 │ ones_like[0][0],  │
│ (Concatenate)       │                   │            │ logical_or[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pre_inject          │ (None, 39, 256)   │          0 │ cnn_reshape[0][0… │
│ (Concatenate)       │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 39)        │          0 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 128)       │    197,120 │ pre_inject[0][0], │
│                     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 8832)      │  1,139,328 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,121,984 (15.72 MB)

 Trainable params: 4,121,984 (15.72 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
3421/3421 ━━━━━━━━━━━━━━━━━━━━ 241s 69ms/step - accuracy: 0.2780 - loss: 4.2564 - val_accuracy: 0.3086 - val_loss: 3.9622
Epoch 2/20
3421/3421 ━━━━━━━━━━━━━━━━━━━━ 249s 66ms/step - accuracy: 0.3434 - loss: 3.4561 - val_accuracy: 0.3386 - val_loss: 3.7465
Epoch 3/20
3421/3421 ━━━━━━━━━━━━━━━━━━━━ 217s 64ms/step - accuracy: 0.3693 - loss: 3.1513 - val_accuracy: 0.3497 - val_loss: 3.6804
Epoch 4/20
3421/3421 ━━━━━━━━━━━━━━━━━━━━ 222s 65ms/step - accuracy: 0.3857 - loss: 2.9511 - val_accuracy: 0.3545 - val_loss: 3.6783
Epoch 5/20
3421/3421 ━━━━━━━━━━━━━━━━━━━━ 216s 63ms/step - accuracy: 0.4011 - loss: 2.7910 - val_accuracy: 0.3585 - val_loss: 3.6880
Epoch 6/20
3421/3421 ━━━━━━━━━━━━━━━━━━━━ 224s 65ms/step - accuracy: 0.4169 - loss: 2.6510 - val_accuracy: 0.3614 - val_loss: 3.6988
Epoch 7/20
3421/3421 ━━━━━━━━━━━━━━━━━━━━ 221s 65ms/step - accuracy: 0.4303 - loss: 2.5268 - val_accuracy: 0.3612 - val_loss: 3.7308
Epoch 8/20
3421/3421 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.

In [ ]:
# Comparison: semua 6 variasi dalam satu plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('All Variations — Validation Metrics', fontsize=14)

for variant_name, hist in all_histories.items():
    axes[0].plot(hist['val_loss'],     label=variant_name)
    axes[1].plot(hist['val_accuracy'], label=variant_name)

for ax, title in zip(axes, ['Validation Loss', 'Validation Accuracy']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(WEIGHTS_DIR / 'comparison_all_variations.png'), dpi=100, bbox_inches='tight')
plt.show()

# Summary table
print(f'\n{"Variation":<30} {"Final Val Loss":>15} {"Final Val Acc":>15}')
print('-' * 62)
for variant_name, hist in all_histories.items():
    print(
        f'{variant_name:<30}'
        f' {hist["val_loss"][-1]:>15.4f}'
        f' {hist["val_accuracy"][-1]:>15.4f}'
    )

best_variant = min(all_histories, key=lambda k: all_histories[k]['val_loss'][-1])
print(f'\nBest variant (lowest val_loss): {best_variant}')


---
## 6. Evaluation

Load model terbaik berdasarkan val_loss terendah dan jalankan inference pada 5 sample gambar.

In [ ]:
# Override manual jika perlu: best_variant = '1layers_512hidden'

best_n_layers, best_hidden = next(
    (n, h) for (n, h) in VARIATIONS
    if f'{n}layers_{h}hidden' == best_variant
)
print(f'Best variant : {best_variant}  (n_layers={best_n_layers}, hidden_dim={best_hidden})')

best_model = build_model_variant(vocab_size, EMBED_DIM, max_length, best_hidden, best_n_layers)
best_model.load_weights(str(WEIGHTS_DIR / f'lstm_{best_variant}.weights.h5'))
print('Model loaded.')


In [ ]:
# Fallback: load tokenizer/features dari file jika section 3-4 dilewati
if 'tokenizer' not in dir():
    assert TOKENIZER_PATH.exists(), f'Tokenizer not found: {TOKENIZER_PATH}'
    with open(str(TOKENIZER_PATH), 'r', encoding='utf-8') as f:
        tok_json = json.load(f)
    tokenizer  = tokenizer_from_json(json.dumps(tok_json))
    vocab_size = len(tokenizer.word_index) + 1
    print(f'Tokenizer loaded. vocab_size={vocab_size}')

if 'features' not in dir():
    assert FEATURES_PATH.exists(), f'Features not found: {FEATURES_PATH}'
    features = np.load(str(FEATURES_PATH), allow_pickle=True).item()
    print(f'Features loaded. count={len(features)}')

if 'max_length' not in dir():
    raise RuntimeError('max_length not set — run Section 3 first or set manually, e.g. max_length = 34')

index_word = {v: k for k, v in tokenizer.word_index.items()}


In [ ]:
def generate_caption(
    model: Model,
    feature: np.ndarray,
    tokenizer,
    max_length: int,
    index_word: dict,
) -> str:
    """Greedy decoding: predict next word satu per satu hingga <end> atau max_length."""
    in_text = '<start>'
    for _ in range(max_length):
        seq       = tokenizer.texts_to_sequences([in_text])[0]
        seq       = pad_sequences([seq], maxlen=max_length, padding='post')
        pred      = model.predict([feature.reshape(1, -1), seq], verbose=0)
        next_idx  = int(np.argmax(pred[0]))
        next_word = index_word.get(next_idx)
        if next_word is None or next_word == '<end>':
            break
        in_text += ' ' + next_word
    return in_text.replace('<start>', '').strip()


In [ ]:
from PIL import Image
from shared.image_utils import load_image

assert IMAGES_DIR.exists(), f'Images directory not found: {IMAGES_DIR}'
sample_paths = sorted(IMAGES_DIR.glob('*.jpg'))[:5]
assert len(sample_paths) > 0, f'No images found in {IMAGES_DIR}'

_encoder = None  # lazy-loaded jika feature belum di-extract

fig, axes = plt.subplots(1, len(sample_paths), figsize=(4 * len(sample_paths), 5))
if len(sample_paths) == 1:
    axes = [axes]

for ax, img_path in zip(axes, sample_paths):
    img_name = img_path.name

    if img_name in features:
        feature = features[img_name]
    else:
        if _encoder is None:
            print('Building InceptionV3 for on-the-fly extraction...')
            _encoder = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
        img_arr = load_image(str(img_path))
        feature = _encoder.predict(img_arr[np.newaxis, ...], verbose=0)[0]

    caption = generate_caption(best_model, feature, tokenizer, max_length, index_word)

    ax.imshow(Image.open(img_path).convert('RGB'))
    ax.set_title(caption, fontsize=9, wrap=True)
    ax.axis('off')
    print(f'{img_name}: {caption}')

plt.suptitle(f'Generated Captions — {best_variant}', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(str(WEIGHTS_DIR / 'sample_captions.png'), dpi=100, bbox_inches='tight')
plt.show()


---
## 7. BLEU-4 and METEOR Score per Variation

Evaluasi setiap variasi pada test split (last 1000 images).
Metrics: BLEU-4 (corpus), METEOR (average), waktu inferensi.

In [ ]:
import time
import nltk
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score  import corpus_bleu, sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as _meteor_score

smoothie = SmoothingFunction().method4

# ── Test split: last 1000 image IDs that have pre-extracted features ──────
# Requires cleaned_mapping and features from sections 3 and 2
all_image_ids  = list(cleaned_mapping.keys())
test_image_ids = [img_id for img_id in all_image_ids[-1000:] if img_id in features]
print(f'Test split size: {len(test_image_ids)} images')


def evaluate_on_test(caption_fn, test_ids, cleaned_mapping, smoothie):
    """
    caption_fn : callable(img_id) -> caption string
    Returns    : (bleu4, meteor, refs_corpus, hyps_corpus)
    """
    refs_corpus  = []
    hyps_corpus  = []
    meteor_scores = []

    for img_id in test_ids:
        hyp = caption_fn(img_id)
        refs_clean = [
            c.replace('<start>', '').replace('<end>', '').strip()
            for c in cleaned_mapping[img_id]
        ]
        hyp_tokens = hyp.split()
        ref_tokens = [r.split() for r in refs_clean]

        refs_corpus.append(ref_tokens)
        hyps_corpus.append(hyp_tokens)
        meteor_scores.append(_meteor_score(ref_tokens, hyp_tokens))

    bleu4  = corpus_bleu(refs_corpus, hyps_corpus,
                         weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=smoothie)
    meteor = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0.0
    return bleu4, meteor, refs_corpus, hyps_corpus


In [ ]:
scores_per_variation = {}

for n_layers, hidden_dim in VARIATIONS:
    variant_name = f'{n_layers}layers_{hidden_dim}hidden'
    weights_path = WEIGHTS_DIR / f'lstm_{variant_name}.weights.h5'

    if not weights_path.exists():
        print(f'[SKIP] {variant_name} — weights not found at {weights_path}')
        continue

    print(f'Evaluating {variant_name} ...', end=' ', flush=True)
    model = build_model_variant(vocab_size, EMBED_DIM, max_length, hidden_dim, n_layers)
    model.load_weights(str(weights_path))

    t0 = time.time()
    bleu4, meteor, _, _ = evaluate_on_test(
        lambda img_id, m=model: generate_caption(m, features[img_id], tokenizer, max_length, index_word),
        test_image_ids,
        cleaned_mapping,
        smoothie,
    )
    elapsed = time.time() - t0

    scores_per_variation[variant_name] = {
        'bleu4': bleu4, 'meteor': meteor, 'time': elapsed,
        'n_layers': n_layers, 'hidden_dim': hidden_dim,
    }
    print(f'BLEU-4={bleu4:.4f}  METEOR={meteor:.4f}  Time={elapsed:.1f}s')


In [ ]:
print(f'\n{"Variation":<30} {"BLEU-4":>8} {"METEOR":>8} {"Time(s)":>9}')
print('-' * 60)
for vname, s in scores_per_variation.items():
    print(f'{vname:<30} {s["bleu4"]:>8.4f} {s["meteor"]:>8.4f} {s["time"]:>9.1f}')

if scores_per_variation:
    best_bleu_variant   = max(scores_per_variation, key=lambda k: scores_per_variation[k]['bleu4'])
    best_meteor_variant = max(scores_per_variation, key=lambda k: scores_per_variation[k]['meteor'])
    print(f'\nBest BLEU-4  : {best_bleu_variant}   ({scores_per_variation[best_bleu_variant]["bleu4"]:.4f})')
    print(f'Best METEOR  : {best_meteor_variant}   ({scores_per_variation[best_meteor_variant]["meteor"]:.4f})')


---
## 8. Keras vs From Scratch Comparison

Load bobot variasi  ke dalam  (implementasi NumPy from scratch).
Bandingkan BLEU-4, METEOR, dan execution time pada 10 test image yang sama.

In [ ]:
from lstm.layers import LSTMDecoder

# LSTMDecoder supports only a single LSTM layer — use the 1-layer 512-hidden variant
SCRATCH_VARIANT = '1layers_512hidden'
SCRATCH_HIDDEN  = 512
scratch_weights = WEIGHTS_DIR / f'lstm_{SCRATCH_VARIANT}.weights.h5'

assert scratch_weights.exists(), (
    f'Weights not found: {scratch_weights}\n'
    f'Run Section 5 training first.'
)

print(f'Loading {SCRATCH_VARIANT} into Keras model...')
keras_1layer = build_model_variant(vocab_size, EMBED_DIM, max_length, SCRATCH_HIDDEN, 1)
keras_1layer.load_weights(str(scratch_weights))

print('Initializing LSTMDecoder with Keras layer weights...')
decoder = LSTMDecoder(
    lstm_keras_layer       = keras_1layer.get_layer('lstm_1'),
    embedding_keras_layer  = keras_1layer.get_layer('embedding'),
    dense_proj_keras_layer = keras_1layer.get_layer('cnn_proj'),
    dense_out_keras_layer  = keras_1layer.get_layer('output'),
    embed_dim              = EMBED_DIM,
    hidden_dim             = SCRATCH_HIDDEN,
    vocab_size             = vocab_size,
    dense_proj_input       = 2048,
)
print('LSTMDecoder ready.')


def generate_caption_scratch(decoder, feature, tokenizer, max_length, index_word):
    """Generate caption using the NumPy from-scratch LSTMDecoder."""
    start_token = tokenizer.word_index.get('<start>', 1)
    end_token   = tokenizer.word_index.get('<end>',   2)
    tokens = decoder.predict(feature, start_token, end_token, max_length)
    words  = [
        index_word[t] for t in tokens
        if index_word.get(t) not in (None, '<end>', '<start>')
    ]
    return ' '.join(words)


In [ ]:
# Use same 10 test images for both models
compare_ids = test_image_ids[:10]

print(f'Running inference on {len(compare_ids)} test images...')

# ── Keras 1-layer model ─────────────────────────────────────────────────────
keras_hyps = []
t0 = time.time()
for img_id in compare_ids:
    cap = generate_caption(keras_1layer, features[img_id], tokenizer, max_length, index_word)
    keras_hyps.append(cap)
keras_time = time.time() - t0

# ── From-scratch LSTMDecoder ────────────────────────────────────────────────
scratch_hyps = []
t0 = time.time()
for img_id in compare_ids:
    cap = generate_caption_scratch(decoder, features[img_id], tokenizer, max_length, index_word)
    scratch_hyps.append(cap)
scratch_time = time.time() - t0

# ── Compute BLEU-4 and METEOR ───────────────────────────────────────────────
def compute_scores(hyps, img_ids, cleaned_mapping, smoothie):
    refs_corpus  = []
    hyps_corpus  = []
    meteor_scores = []
    for img_id, hyp in zip(img_ids, hyps):
        refs_clean = [
            c.replace('<start>', '').replace('<end>', '').strip()
            for c in cleaned_mapping[img_id]
        ]
        ref_tokens = [r.split() for r in refs_clean]
        hyp_tokens = hyp.split()
        refs_corpus.append(ref_tokens)
        hyps_corpus.append(hyp_tokens)
        meteor_scores.append(_meteor_score(ref_tokens, hyp_tokens))
    bleu4  = corpus_bleu(refs_corpus, hyps_corpus,
                         weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=smoothie)
    meteor = sum(meteor_scores) / len(meteor_scores)
    return bleu4, meteor

keras_bleu4,   keras_meteor   = compute_scores(keras_hyps,   compare_ids, cleaned_mapping, smoothie)
scratch_bleu4, scratch_meteor = compute_scores(scratch_hyps, compare_ids, cleaned_mapping, smoothie)

print(f'\nKeras   ({SCRATCH_VARIANT}): BLEU-4={keras_bleu4:.4f}  METEOR={keras_meteor:.4f}  Time={keras_time:.2f}s')
print(f'Scratch ({SCRATCH_VARIANT}): BLEU-4={scratch_bleu4:.4f}  METEOR={scratch_meteor:.4f}  Time={scratch_time:.2f}s')


In [ ]:
print(f'\n{"Model":<35} {"BLEU-4":>8} {"METEOR":>8} {"Time(s)":>9}')
print('-' * 65)
print(f'{"Keras — " + SCRATCH_VARIANT:<35} {keras_bleu4:>8.4f} {keras_meteor:>8.4f} {keras_time:>9.2f}')
print(f'{"From Scratch (LSTMDecoder)":<35} {scratch_bleu4:>8.4f} {scratch_meteor:>8.4f} {scratch_time:>9.2f}')

bleu_delta   = scratch_bleu4  - keras_bleu4
meteor_delta = scratch_meteor - keras_meteor
time_ratio   = scratch_time   / keras_time if keras_time > 0 else float('inf')
print(f'\nDelta (Scratch - Keras): BLEU-4={bleu_delta:+.4f}  METEOR={meteor_delta:+.4f}')
print(f'Speed ratio (Scratch / Keras): {time_ratio:.1f}x')


---
## 9. Caption Length Variation

Evaluasi pengaruh  generasi terhadap kualitas caption (BLEU-4).
Model tetap menggunakan best Keras model; hanya jumlah langkah generasi yang divariasikan.

In [ ]:
TRAINED_MAX_LENGTH = max_length   # fixed pad length used during training
MAX_LENGTH_VARIANTS = [10, 20, 34]

def generate_caption_maxsteps(model, feature, tokenizer, pad_length, max_steps, index_word):
    """
    Greedy decoding with separate pad_length (for input shape) and max_steps (stop criterion).
    pad_length must match the value the model was trained with.
    max_steps controls how many tokens to generate at most.
    """
    in_text = '<start>'
    for _ in range(max_steps):
        seq       = tokenizer.texts_to_sequences([in_text])[0]
        seq       = pad_sequences([seq], maxlen=pad_length, padding='post')
        pred      = model.predict([feature.reshape(1, -1), seq], verbose=0)
        next_idx  = int(np.argmax(pred[0]))
        next_word = index_word.get(next_idx)
        if next_word is None or next_word == '<end>':
            break
        in_text += ' ' + next_word
    return in_text.replace('<start>', '').strip()


length_scores = {}

for ml in MAX_LENGTH_VARIANTS:
    print(f'Evaluating max_steps={ml} ...', end=' ', flush=True)
    bleu4, _, _, _ = evaluate_on_test(
        lambda img_id, ml=ml: generate_caption_maxsteps(
            best_model, features[img_id], tokenizer, TRAINED_MAX_LENGTH, ml, index_word
        ),
        test_image_ids,
        cleaned_mapping,
        smoothie,
    )
    length_scores[ml] = bleu4
    print(f'BLEU-4={bleu4:.4f}')


In [ ]:
ml_values   = list(length_scores.keys())
bleu_values = list(length_scores.values())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ml_values, bleu_values, marker='o', linewidth=2, markersize=8)
for x, y in zip(ml_values, bleu_values):
    ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(0, 10), ha='center')
ax.set_xlabel('max_length (generation steps)')
ax.set_ylabel('BLEU-4')
ax.set_title(f'BLEU-4 vs Caption Max Length — {best_variant}')
ax.set_xticks(ml_values)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(WEIGHTS_DIR / 'bleu4_vs_maxlength.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'\n{"max_length":>12} {"BLEU-4":>10}')
print('-' * 25)
for ml, b in length_scores.items():
    print(f'{ml:>12} {b:>10.4f}')

best_ml = max(length_scores, key=length_scores.get)
print(f'\nBest max_length: {best_ml}  (BLEU-4={length_scores[best_ml]:.4f})')


---
## 10. Qualitative Analysis

Pilih 10 test image dengan skor BLEU-4 bervariasi (tinggi, sedang, rendah).
Tampilkan gambar, caption Keras, caption From Scratch, ground truth, dan BLEU-4.

In [ ]:
from PIL import Image

print('Computing per-image BLEU-4 scores on test split...')
per_image_data = []

for img_id in test_image_ids:
    feature    = features[img_id]
    keras_cap  = generate_caption(best_model, feature, tokenizer, max_length, index_word)
    refs_clean = [
        c.replace('<start>', '').replace('<end>', '').strip()
        for c in cleaned_mapping[img_id]
    ]
    ref_tokens = [r.split() for r in refs_clean]
    hyp_tokens = keras_cap.split()

    img_bleu4 = sentence_bleu(ref_tokens, hyp_tokens,
                               weights=(0.25, 0.25, 0.25, 0.25),
                               smoothing_function=smoothie)
    per_image_data.append({
        'img_id':    img_id,
        'keras_cap': keras_cap,
        'refs':      refs_clean,
        'bleu4':     img_bleu4,
    })

# Sort by BLEU-4 and sample high / medium / low examples
sorted_data = sorted(per_image_data, key=lambda x: x['bleu4'])
n = len(sorted_data)

low_samples    = sorted_data[:3]                         # lowest 3
mid_samples    = sorted_data[n // 2 - 2 : n // 2 + 2]   # 4 around median
high_samples   = sorted_data[-3:]                        # highest 3

selected = (low_samples + mid_samples + high_samples)[:10]
print(f'Selected {len(selected)} images: 3 low / 4 mid / 3 high BLEU-4')
print(f'BLEU-4 range: {selected[0]["bleu4"]:.4f} (low) to {selected[-1]["bleu4"]:.4f} (high)')


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
axes = axes.flatten()

for ax, item in zip(axes, selected):
    img_id  = item['img_id']
    img_path = IMAGES_DIR / img_id

    # ── Generate from-scratch caption ──────────────────────────────────
    scratch_cap = generate_caption_scratch(
        decoder, features[img_id], tokenizer, max_length, index_word
    )

    # ── Display image ───────────────────────────────────────────────────
    if img_path.exists():
        ax.imshow(Image.open(img_path).convert('RGB'))
    else:
        ax.set_facecolor('#cccccc')
        ax.text(0.5, 0.5, 'image\nnot found', ha='center', va='center',
                transform=ax.transAxes, fontsize=8)
    ax.axis('off')

    # ── Annotate ────────────────────────────────────────────────────────
    gt_lines = '\n'.join(f'GT: {r[:55]}' for r in item['refs'][:2])
    title = (
        f'[Keras]   {item["keras_cap"][:55]}\n'
        f'[Scratch] {scratch_cap[:55]}\n'
        f'{gt_lines}\n'
        f'BLEU-4: {item["bleu4"]:.4f}'
    )
    ax.set_title(title, fontsize=7, loc='left', pad=3)

plt.suptitle(
    f'Qualitative Analysis — Best Keras ({best_variant}) vs From Scratch ({SCRATCH_VARIANT})',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig(str(WEIGHTS_DIR / 'qualitative_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

# ── Print summary ───────────────────────────────────────────────────────────
print(f'\n{"Image ID":<40} {"BLEU-4":>8}  Keras Caption (truncated)')
print('-' * 90)
for item in selected:
    print(f'{item["img_id"]:<40} {item["bleu4"]:>8.4f}  {item["keras_cap"][:40]}')
